# Flight price tracker
Run cells top to bottom. Start with the mock provider, then switch to Amadeus.

In [ ]:
import os, time
from flighttracker.db import DB
from flighttracker.providers import get_provider
from flighttracker.notify import NOTIFIERS
from flighttracker.cli import run_checks

db = DB("flights.db")  # price history persists here between runs

## Settings (optional)
Fill in for real prices / phone alerts.

In [ ]:
PROVIDER = "mock"        # change to "amadeus" for real prices
NOTIFY = ["console"]     # add "ntfy" and/or "email"

# os.environ["AMADEUS_CLIENT_ID"] = "..."
# os.environ["AMADEUS_CLIENT_SECRET"] = "..."
# os.environ["AMADEUS_ENV"] = "production"
# os.environ["NTFY_TOPIC"] = "my-flight-alerts"

provider = get_provider(PROVIDER)
notifiers = [NOTIFIERS[n] for n in NOTIFY]

## Add a route

In [ ]:
db.add_watch("ORD", "LHR", "2027-01-10", return_date="2027-01-20")
[dict(w) for w in db.watches()]

## Check once

In [ ]:
run_checks(db, provider, notifiers, min_history=3)

## Check repeatedly
This cell keeps running while the notebook is open; use Interrupt (stop button) to end it.

In [ ]:
INTERVAL = 3600  # seconds (use 2 with the mock provider to see alerts quickly)
try:
    while True:
        run_checks(db, provider, notifiers, min_history=3)
        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("stopped")

## Price history chart

In [ ]:
import matplotlib.pyplot as plt
for w in db.watches():
    h = db.history(w["id"])
    plt.plot([r["price"] for r in h], marker="o", label=f'{w["origin"]}-{w["destination"]} {w["depart_date"]}')
plt.xlabel("check #"); plt.ylabel("price"); plt.legend(); plt.show()